# Modelo de Clasificación Multi-clase: 3 Clases

Este notebook entrena un modelo de clasificación usando **3 clases clínicas**:
- `bajo` → Precision < 35
- `medio` → 35 ≤ Precision < 60
- `alto` → Precision ≥ 60

Se comparan múltiples algoritmos y se selecciona el mejor basado en validación.

In [ ]:
# Importar bibliotecas y configuración
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_PATH = Path('/Users/cristobal/Desktop/Capstone/neuroflex_dataset_augmented.csv')

print('Configuración inicial lista.')
print('Notebook: Modelo de 3 Clases')

In [ ]:
# Cargar y explorar el dataset
df = pd.read_csv(DATA_PATH)
print(f'Forma del dataset: {df.shape}')
print('\nColumnas disponibles:')
print(df.columns.tolist()[:10], '... total:', len(df.columns))

# Limpiar datos
if 'age' in df.columns:
    df['age'] = pd.to_numeric(df['age'], errors='coerce').fillna(0).round().astype(int)

if 'Precision' in df.columns:
    df['Precision'] = pd.to_numeric(df['Precision'], errors='coerce').clip(lower=0, upper=100)

print('\nValores nulos por columna:')
null_counts = df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False)
if len(null_counts) > 0:
    print(null_counts)
else:
    print('No hay valores nulos')

In [ ]:
# Crear variable objetivo con 3 clases
df['performance_band'] = pd.cut(
    df['Precision'],
    bins=[-1, 35, 60, 101],
    labels=['bajo', 'medio', 'alto'],
    right=False,
    include_lowest=True,
)
df['target'] = df['performance_band'].map({'bajo': 0, 'medio': 1, 'alto': 2})

print('Distribución de la variable objetivo (3 clases):')
print(df['target'].value_counts(normalize=True).sort_index())
print(f"\nCuentas absolutas:")
print(df['target'].value_counts().sort_index())

# Visualizar distribución
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['Precision'], bins=30, edgecolor='black', color='steelblue')
axes[0].axvline(35, color='red', linestyle='--', linewidth=2, label='Umbral 35')
axes[0].axvline(60, color='orange', linestyle='--', linewidth=2, label='Umbral 60')
axes[0].set_title('Distribución de Precision')
axes[0].set_xlabel('Precision')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

df['target'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_title('Distribución de Clases Objetivo (3 clases)')
axes[1].set_xlabel('Clase')
axes[1].set_ylabel('Frecuencia')
axes[1].set_xticklabels(['bajo (0)', 'medio (1)', 'alto (2)'], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Feature engineering
model_df = df.copy()

# Eliminar columnas irrelevantes
exclude_cols = ['Performance', 'rendimiento', 'performance_band']
model_df = model_df.drop(columns=[c for c in exclude_cols if c in model_df.columns], errors='ignore')

# Crear nuevas características
model_df['Precision_bin'] = pd.cut(
    model_df['Precision'], 
    bins=[0, 25, 50, 75, 100], 
    labels=['muy_baja', 'baja', 'media', 'alta'], 
    include_lowest=True
)
model_df['age_group'] = pd.cut(
    model_df['age'], 
    bins=[0, 40, 60, 80, 120], 
    labels=['joven', 'adulto', 'mayor', 'anciano'], 
    include_lowest=True
)

model_df['tiempo_total'] = (
    model_df.get('TiempoRespuestaPararse', 0) + 
    model_df.get('TiempoActivoTarea', 0) + 
    model_df.get('TiempoTutorial', 0)
)
model_df['ratio_aciertos'] = (
    model_df.get('CantAciertasTotales', 0) / 
    (model_df.get('ObjetosInteractuadosCorrectamente', 0) + 1)
)

# Transformaciones logarítmicas
for col in ['TiempoRespuestaPararse', 'TiempoActivoTarea', 'TiempoTutorial', 
            'TiempoRespuestaPregunta1', 'TiempoRespuestaPregunta2', 'TiempoRespuestaPregunta3']:
    if col in model_df.columns:
        model_df[f'{col}_log'] = np.log1p(model_df[col].clip(lower=0))

# Seleccionar features
feature_cols = [c for c in model_df.columns if c not in ['target', 'performance_band', 'Precision']]
print('Número de features finales:', len(feature_cols))

In [ ]:
# Dividir datos en train, validation y test
X_final = model_df[feature_cols]
y_final = model_df['target']

X_train, X_temp, y_train, y_temp = train_test_split(
    X_final, y_final,
    test_size=0.30,
    random_state=SEED,
    stratify=y_final,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print('Train size:', X_train.shape)
print('Validation size:', X_val.shape)
print('Test size:', X_test.shape)
print('\nDistribución de clases (train):')
print(pd.Series(y_train).value_counts().sort_index())
print('\nDistribución de clases (validation):')
print(pd.Series(y_val).value_counts().sort_index())
print('\nDistribución de clases (test):')
print(pd.Series(y_test).value_counts().sort_index())

In [ ]:
# Crear preprocesador
categorical_cols = [c for c in X_train.columns if X_train[c].dtype == 'object' or pd.api.types.is_categorical_dtype(X_train[c])]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols and pd.api.types.is_numeric_dtype(X_train[c])]

preprocessor_model = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
    ]
)

print('Preprocesador creado.')
print(f'Columnas numéricas: {len(numeric_cols)}')
print(f'Columnas categóricas: {len(categorical_cols)}')

In [ ]:
# Entrenar múltiples modelos y compararlos
models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED),
    'RandomForest': RandomForestClassifier(n_estimators=500, random_state=SEED, class_weight='balanced'),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=500, random_state=SEED, class_weight='balanced'),
    'GradientBoosting': GradientBoostingClassifier(random_state=SEED),
    'SVC': SVC(class_weight='balanced', random_state=SEED, probability=True),
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor_model),
        ('model', model),
    ])
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)
    
    val_acc = accuracy_score(y_val, val_pred)
    val_prec = precision_score(y_val, val_pred, average='weighted', zero_division=0)
    val_rec = recall_score(y_val, val_pred, average='weighted', zero_division=0)
    val_f1 = f1_score(y_val, val_pred, average='weighted', zero_division=0)
    
    results.append({
        'model': name,
        'accuracy': val_acc,
        'precision': val_prec,
        'recall': val_rec,
        'f1': val_f1,
    })
    
    print(f'\n=== {name} ===')
    print(f'Accuracy: {val_acc:.4f}')
    print(f'Precision (weighted): {val_prec:.4f}')
    print(f'Recall (weighted): {val_rec:.4f}')
    print(f'F1 (weighted): {val_f1:.4f}')

results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
print('\n' + '='*60)
print('Ranking por F1 en validación:')
print('='*60)
print(results_df)

In [ ]:
# Visualizar comparación de modelos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

results_df.plot(x='model', y='accuracy', kind='barh', ax=axes[0, 0], color='steelblue', legend=False)
axes[0, 0].set_title('Accuracy en Validación')
axes[0, 0].set_xlabel('Accuracy')

results_df.plot(x='model', y='precision', kind='barh', ax=axes[0, 1], color='coral', legend=False)
axes[0, 1].set_title('Precision (weighted) en Validación')
axes[0, 1].set_xlabel('Precision')

results_df.plot(x='model', y='recall', kind='barh', ax=axes[1, 0], color='lightgreen', legend=False)
axes[1, 0].set_title('Recall (weighted) en Validación')
axes[1, 0].set_xlabel('Recall')

results_df.plot(x='model', y='f1', kind='barh', ax=axes[1, 1], color='gold', legend=False)
axes[1, 1].set_title('F1-Score (weighted) en Validación')
axes[1, 1].set_xlabel('F1-Score')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluar el mejor modelo en test
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f'\n{"="*60}')
    print(f'EVALUACIÓN FINAL EN TEST: {model_name}')
    print(f'{"="*60}')
    print(f'Accuracy: {acc:.4f}')
    print(f'Precision (weighted): {precision:.4f}')
    print(f'Recall (weighted): {recall:.4f}')
    print(f'F1-Score (weighted): {f1:.4f}')
    print('\nMatriz de Confusión:')
    print(cm)
    
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm,
        'y_pred': y_pred,
    }

# Seleccionar y evaluar el mejor modelo
best_model_name = results_df.iloc[0]['model']
best_model = models[best_model_name]

best_pipe = Pipeline([
    ('preprocessor', preprocessor_model),
    ('model', best_model),
])

print(f'\n✓ Modelo seleccionado: {best_model_name}')
final_metrics = evaluate_model(best_pipe, X_train, y_train, X_test, y_test, best_model_name)

In [ ]:
# Visualizar matriz de confusión y reporte detallado
y_test_pred = final_metrics['y_pred']
cm = final_metrics['confusion_matrix']

# Matriz de confusión
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['bajo', 'medio', 'alto'],
            yticklabels=['bajo', 'medio', 'alto'],
            cbar_kws={'label': 'Frecuencia'})
plt.title(f'Matriz de Confusión - Modelo: {best_model_name}\nConjunto Test')
plt.ylabel('Clase Real')
plt.xlabel('Clase Predicha')
plt.tight_layout()
plt.show()

# Reporte de clasificación detallado
print('\n' + '='*60)
print('REPORTE DETALLADO DE CLASIFICACIÓN')
print('='*60)
print(classification_report(y_test, y_test_pred, 
                          target_names=['bajo', 'medio', 'alto'],
                          digits=4))

## Conclusiones

**Modelo de 3 Clases Clínicas:**
- Permite capturar más nuances en el rendimiento cognitivo
- Proporciona información clínica más detallada
- Puede tener accuracy ligeramente más bajo que una clasificación binaria (debido a la complejidad)
- Es más útil si se necesita diferenciar entre rendimiento bajo, medio y alto

**Thresholds utilizados:**
- `bajo` → Precision < 35
- `medio` → 35 ≤ Precision < 60  
- `alto` → Precision ≥ 60

Compara este modelo con el de 2 clases en el notebook `neuroflex_comparison_2vs3classes.ipynb` para entender las trade-offs.

## Conclusiones

**Modelo de 3 Clases Clínicas:**
- Permite capturar más nuances en el rendimiento cognitivo
- Proporciona información clínica más detallada
- Puede tener accuracy ligeramente más bajo que una clasificación binaria (debido a la complejidad)
- Es más útil si se necesita diferenciar entre rendimiento bajo, medio y alto

**Thresholds utilizados:**
- `bajo` → Precision < 35
- `medio` → 35 ≤ Precision < 60  
- `alto` → Precision ≥ 60

Compara este modelo con el de 2 clases en el notebook `neuroflex_comparison_2vs3classes.ipynb` para entender las trade-offs.

In [ ]:
# Visualizar matriz de confusión y reporte detallado
y_test_pred = final_metrics['y_pred']
cm = final_metrics['confusion_matrix']

# Matriz de confusión
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['bajo', 'medio', 'alto'],
            yticklabels=['bajo', 'medio', 'alto'],
            cbar_kws={'label': 'Frecuencia'})
plt.title(f'Matriz de Confusión - Modelo: {best_model_name}\nConjunto Test')
plt.ylabel('Clase Real')
plt.xlabel('Clase Predicha')
plt.tight_layout()
plt.show()

# Reporte de clasificación detallado
print('\n' + '='*60)
print('REPORTE DETALLADO DE CLASIFICACIÓN')
print('='*60)
print(classification_report(y_test, y_test_pred, 
                          target_names=['bajo', 'medio', 'alto'],
                          digits=4))

In [ ]:
# Evaluar el mejor modelo en test
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f'\n{"="*60}')
    print(f'EVALUACIÓN FINAL EN TEST: {model_name}')
    print(f'{"="*60}')
    print(f'Accuracy: {acc:.4f}')
    print(f'Precision (weighted): {precision:.4f}')
    print(f'Recall (weighted): {recall:.4f}')
    print(f'F1-Score (weighted): {f1:.4f}')
    print('\nMatriz de Confusión:')
    print(cm)
    
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm,
        'y_pred': y_pred,
    }

# Seleccionar y evaluar el mejor modelo
best_model_name = results_df.iloc[0]['model']
best_model = models[best_model_name]

best_pipe = Pipeline([
    ('preprocessor', preprocessor_model),
    ('model', best_model),
])

print(f'\n✓ Modelo seleccionado: {best_model_name}')
final_metrics = evaluate_model(best_pipe, X_train, y_train, X_test, y_test, best_model_name)

In [ ]:
# Visualizar comparación de modelos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

results_df.plot(x='model', y='accuracy', kind='barh', ax=axes[0, 0], color='steelblue', legend=False)
axes[0, 0].set_title('Accuracy en Validación')
axes[0, 0].set_xlabel('Accuracy')

results_df.plot(x='model', y='precision', kind='barh', ax=axes[0, 1], color='coral', legend=False)
axes[0, 1].set_title('Precision (weighted) en Validación')
axes[0, 1].set_xlabel('Precision')

results_df.plot(x='model', y='recall', kind='barh', ax=axes[1, 0], color='lightgreen', legend=False)
axes[1, 0].set_title('Recall (weighted) en Validación')
axes[1, 0].set_xlabel('Recall')

results_df.plot(x='model', y='f1', kind='barh', ax=axes[1, 1], color='gold', legend=False)
axes[1, 1].set_title('F1-Score (weighted) en Validación')
axes[1, 1].set_xlabel('F1-Score')

plt.tight_layout()
plt.show()

In [ ]:
# Entrenar múltiples modelos y compararlos
models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED),
    'RandomForest': RandomForestClassifier(n_estimators=500, random_state=SEED, class_weight='balanced'),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=500, random_state=SEED, class_weight='balanced'),
    'GradientBoosting': GradientBoostingClassifier(random_state=SEED),
    'SVC': SVC(class_weight='balanced', random_state=SEED, probability=True),
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor_model),
        ('model', model),
    ])
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)
    
    val_acc = accuracy_score(y_val, val_pred)
    val_prec = precision_score(y_val, val_pred, average='weighted', zero_division=0)
    val_rec = recall_score(y_val, val_pred, average='weighted', zero_division=0)
    val_f1 = f1_score(y_val, val_pred, average='weighted', zero_division=0)
    
    results.append({
        'model': name,
        'accuracy': val_acc,
        'precision': val_prec,
        'recall': val_rec,
        'f1': val_f1,
    })
    
    print(f'\n=== {name} ===')
    print(f'Accuracy: {val_acc:.4f}')
    print(f'Precision (weighted): {val_prec:.4f}')
    print(f'Recall (weighted): {val_rec:.4f}')
    print(f'F1 (weighted): {val_f1:.4f}')

results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
print('\n' + '='*60)
print('Ranking por F1 en validación:')
print('='*60)
print(results_df)

In [ ]:
# Crear preprocesador
categorical_cols = [c for c in X_train.columns if X_train[c].dtype == 'object' or pd.api.types.is_categorical_dtype(X_train[c])]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols and pd.api.types.is_numeric_dtype(X_train[c])]

preprocessor_model = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
    ]
)

print('Preprocesador creado.')
print(f'Columnas numéricas: {len(numeric_cols)}')
print(f'Columnas categóricas: {len(categorical_cols)}')

In [ ]:
# Dividir datos en train, validation y test
X_final = model_df[feature_cols]
y_final = model_df['target']

X_train, X_temp, y_train, y_temp = train_test_split(
    X_final, y_final,
    test_size=0.30,
    random_state=SEED,
    stratify=y_final,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print('Train size:', X_train.shape)
print('Validation size:', X_val.shape)
print('Test size:', X_test.shape)
print('\nDistribución de clases (train):')
print(pd.Series(y_train).value_counts().sort_index())
print('\nDistribución de clases (validation):')
print(pd.Series(y_val).value_counts().sort_index())
print('\nDistribución de clases (test):')
print(pd.Series(y_test).value_counts().sort_index())

In [ ]:
# Feature engineering
model_df = df.copy()

# Eliminar columnas irrelevantes
exclude_cols = ['Performance', 'rendimiento', 'performance_band']
model_df = model_df.drop(columns=[c for c in exclude_cols if c in model_df.columns], errors='ignore')

# Crear nuevas características
model_df['Precision_bin'] = pd.cut(
    model_df['Precision'], 
    bins=[0, 25, 50, 75, 100], 
    labels=['muy_baja', 'baja', 'media', 'alta'], 
    include_lowest=True
)
model_df['age_group'] = pd.cut(
    model_df['age'], 
    bins=[0, 40, 60, 80, 120], 
    labels=['joven', 'adulto', 'mayor', 'anciano'], 
    include_lowest=True
)

model_df['tiempo_total'] = (
    model_df.get('TiempoRespuestaPararse', 0) + 
    model_df.get('TiempoActivoTarea', 0) + 
    model_df.get('TiempoTutorial', 0)
)
model_df['ratio_aciertos'] = (
    model_df.get('CantAciertasTotales', 0) / 
    (model_df.get('ObjetosInteractuadosCorrectamente', 0) + 1)
)

# Transformaciones logarítmicas
for col in ['TiempoRespuestaPararse', 'TiempoActivoTarea', 'TiempoTutorial', 
            'TiempoRespuestaPregunta1', 'TiempoRespuestaPregunta2', 'TiempoRespuestaPregunta3']:
    if col in model_df.columns:
        model_df[f'{col}_log'] = np.log1p(model_df[col].clip(lower=0))

# Seleccionar features
feature_cols = [c for c in model_df.columns if c not in ['target', 'performance_band', 'Precision']]
print('Número de features finales:', len(feature_cols))

In [ ]:
# Crear variable objetivo con 3 clases
df['performance_band'] = pd.cut(
    df['Precision'],
    bins=[-1, 35, 60, 101],
    labels=['bajo', 'medio', 'alto'],
    right=False,
    include_lowest=True,
)
df['target'] = df['performance_band'].map({'bajo': 0, 'medio': 1, 'alto': 2})

print('Distribución de la variable objetivo (3 clases):')
print(df['target'].value_counts(normalize=True).sort_index())
print(f"\nCuentas absolutas:")
print(df['target'].value_counts().sort_index())

# Visualizar distribución
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['Precision'], bins=30, edgecolor='black', color='steelblue')
axes[0].axvline(35, color='red', linestyle='--', linewidth=2, label='Umbral 35')
axes[0].axvline(60, color='orange', linestyle='--', linewidth=2, label='Umbral 60')
axes[0].set_title('Distribución de Precision')
axes[0].set_xlabel('Precision')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

df['target'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_title('Distribución de Clases Objetivo (3 clases)')
axes[1].set_xlabel('Clase')
axes[1].set_ylabel('Frecuencia')
axes[1].set_xticklabels(['bajo (0)', 'medio (1)', 'alto (2)'], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Cargar y explorar el dataset
df = pd.read_csv(DATA_PATH)
print(f'Forma del dataset: {df.shape}')
print('\nColumnas disponibles:')
print(df.columns.tolist()[:10], '... total:', len(df.columns))

# Limpiar datos
if 'age' in df.columns:
    df['age'] = pd.to_numeric(df['age'], errors='coerce').fillna(0).round().astype(int)

if 'Precision' in df.columns:
    df['Precision'] = pd.to_numeric(df['Precision'], errors='coerce').clip(lower=0, upper=100)

print('\nValores nulos por columna:')
null_counts = df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False)
if len(null_counts) > 0:
    print(null_counts)
else:
    print('No hay valores nulos')

In [ ]:
# Importar bibliotecas y configuración
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_PATH = Path('/Users/cristobal/Desktop/Capstone/neuroflex_dataset_augmented.csv')

print('Configuración inicial lista.')
print('Notebook: Modelo de 3 Clases')

# Modelo de Clasificación Multi-clase: 3 Clases

Este notebook entrena un modelo de clasificación usando **3 clases clínicas**:
- `bajo` → Precision < 35
- `medio` → 35 ≤ Precision < 60
- `alto` → Precision ≥ 60

Se comparan múltiples algoritmos y se selecciona el mejor basado en validación.